# Examen de diseño de red 5G

**Alumno:** _________________ · **Mapa asignado:** `jesus-maria-01` · **Duración:** 3 h

**Material permitido:** los apuntes de la lección
([Sesión 07](https://ollerenac.github.io/wireless-communication-systems/sessions/07-network-design/))
y este notebook. Las **pistas** de cada fase te dicen qué tabla o sección
consultar.

**Cómo se califica:** cada número que escribas debe tener **origen**: una
oferta de la publicidad, una cláusula del contrato, o una tabla de la
lección (citada). Un valor distinto al de la pauta pero bien defendido; un valor "correcto" sin defender, no.

**Dinámica de cada fase** — 4 bloques:

1. **Enunciado**: la parte del proyecto que esta fase resuelve.
2. **Pista**: qué tabla/sección de la lección consultar.
3. **Celda `TU TRABAJO`**: complétala (los `None` son tuyos).
4. **Celda `VERIFICADOR`**: ejecútala sin modificarla — chequea que tu
   respuesta esté bien *formada* (no que esté bien *pensada*).
5. **Celda `JUSTIFICACIÓN`**: responde en 2–3 líneas por pregunta.

---


In [4]:
# ---- Preparación del entorno (ejecutar SIEMPRE esta celda primero) ----
# ESCENA: el mapa asignado para tu examen.
ESCENA = "jesus-maria-01"

import importlib.util, os

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ModuleNotFoundError:
    EN_COLAB = False

if EN_COLAB:
    if importlib.util.find_spec("sionna") is None:
        %pip install -q sionna-rt
    if not os.path.exists(f"escenas/{ESCENA}/{ESCENA}.xml"):
        !wget -q https://ollerenac.github.io/wireless-communication-systems/sessions/07-network-design/escenas/{ESCENA}.zip
        !unzip -qo {ESCENA}.zip -d escenas && rm {ESCENA}.zip
    os.makedirs("resultados", exist_ok=True)
    print(f"Colab listo: Sionna RT instalado, mapa '{ESCENA}' descargado.")
else:
    print(f"entorno local (kernel ran-design) — mapa '{ESCENA}' del repo")
assert os.path.exists(f"escenas/{ESCENA}/{ESCENA}.xml"), \
    f"no se encontró escenas/{ESCENA}/{ESCENA}.xml — ¿nombre correcto?"


entorno local (kernel ran-design) — mapa 'jesus-maria-01' del repo



## El proyecto

Un operador móvil entrante te contrata para diseñar su red de acceso 5G
en un polígono de **Jesús María, Lima** — 1.27 × 1.07 km de distrito
residencial denso, con el eje hospitalario de la Av. Arenales (Torre
Trecca, Edificio Lima) dentro del área. Su publicidad ya está impresa:

> *"5G real en todo Jesús María. Video HD sin cortes y videollamadas que
> no se caen, en tu casa y en la calle. La red que sí llega."*

Términos de referencia (TdR):

- Licencia: **80 MHz en n78** (3.5 GHz), TDD.
- Azoteas disponibles; el municipio autoriza **máximo 7 sitios**.
- Potencia máxima por sector: **43 dBm**.
- Market: ~**30 000 hab/km²**, participación objetivo **25%**,
  consumo típico **12 GB/mes** por abonado.


---

## Fase 0 — Requisitos: de la publicidad al contrato de diseño

**Enunciado.** Traduce la publicidad y los términos de referencia a un
diccionario `REQ` medible. Toda oferta debe convertirse en **número con
probabilidad**; todo número que declares, el trazador lo va a cobrar en
la Fase 6.

**Pistas.**

- *"en todo Jesús María"* → ¿qué umbral RSRP y qué probabilidad? →
  **Tabla 0.1** y **§0.1** (umbral y probabilidad de control).
- *¿cuánto SINR necesitan los datos y por qué ese umbral?* → **Tabla 0.2**
  y **§0.2** (umbral y probabilidad de datos).
- *"video HD sin cortes"* → ¿qué bitrate por usuario? → **Tabla 0.3** (§0.3) —
  y ojo: el throughput de **borde** no se deriva de la demanda — se
  **adopta**: piso en esta tabla, referencia en los valores publicados de la industria,
  adopción declarada (§0.3, los tres pasos). El UL no tiene referencia
  publicada: sale de la **columna UL** sobre los servicios que TU
  publicidad ofrece, más un margen que tú declaras (§0.3).
- Capacidad agregada → la cadena GB/mes → kbps de hora cargada está en
  **§0.4** (Tabla 0.4 la tabula para volúmenes típicos).
- La estructura de un `REQ` completo (qué requisitos existen) → **Tabla
  0.5** (§0.5).


In [ ]:
# ================= TU TRABAJO — completa cada None =================
# Cada número debe tener origen (publicidad, contrato, o tabla citada).

REQ = {
    # R1 — área de servicio: el mapa asignado (la escena no la toques)
    "escena":            f"escenas/{ESCENA}/{ESCENA}.xml",
    "area_km2":          None,   # dimensiones del mapa
    # R2 — cobertura del plano de control: "en todo Jesús María"
    "rsrp_min_dbm":      None,   # sección 0.1
    "rsrp_prob":         None,   # sección 0.1
    # R3 — calidad de datos: "video HD sin cortes"
    "sinr_min_db":       None,   # sección 0.2
    "sinr_prob":         None,   # sección 0.2   # fracción (0 a 1)
    # R4 — throughput de borde (percentil 5 del área)
    "thr_borde_dl_mbps": None,   # sección 0.3
    "thr_borde_ul_mbps": None,   # sección 0.3
    # R5 — capacidad agregada en hora cargada (completa el cálculo abajo)
    "capacidad_mbps_km2": None,  # sección 0.4
    # (R6 — servicios y latencia: se garantiza por arquitectura/QoS,
    #  no hay mapa del trazador que lo verifique -> no entra al dict)
    # R7 — espectro licenciado
    "banda":             None,   # texto, p. ej. "n78"
    "fc_hz":             None,
    "bw_hz":             None,
    # R8 — restricciones de despliegue
    "max_sitios":        None,
    "p_tx_dbm_max":      None,
}

# --- El cálculo detrás de R5 (fórmula en §0.4) ---
personas_km2     = None   # de los TdR
market_share     = None   # de los TdR (fracción)
gb_mes           = None   # de los TdR
f_bh             = None   # fracción de tráfico en hora cargada (§0.4: rango típico)

kbps_por_abonado = None   # escribe aquí tu cálculo con la fórmula de §0.4
demanda_mbps_km2 = None   # abonados/km2 x kbps_por_abonado, en Mbps/km2

if None in (kbps_por_abonado, demanda_mbps_km2, REQ["capacidad_mbps_km2"]):
    print("(hay None pendientes — completa y vuelve a ejecutar)")
else:
    print(f"{kbps_por_abonado:.0f} kbps/abonado -> demanda {demanda_mbps_km2:.0f} Mbps/km2 "
          f"(tu requisito R5: {REQ['capacidad_mbps_km2']:.0f})")


(hay None pendientes — completa y vuelve a ejecutar)


In [ ]:
# ============ VERIFICADOR — ejecuta esta celda SIN modificarla ============
# Chequea que tu REQ esté bien FORMADO. No chequea que esté bien PENSADO:
# un valor absurdo puede pasar aquí y reprobar en la Fase 6.

CLAVES = {"escena", "area_km2", "rsrp_min_dbm", "rsrp_prob", "sinr_min_db",
          "sinr_prob", "thr_borde_dl_mbps", "thr_borde_ul_mbps",
          "capacidad_mbps_km2", "banda", "fc_hz", "bw_hz",
          "max_sitios", "p_tx_dbm_max"}

faltan = CLAVES - set(REQ)
assert not faltan, f"faltan claves en REQ: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES if REQ.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# rangos de cordura (anchos a propósito — no son la respuesta)
assert -140 < REQ["rsrp_min_dbm"] < -60,  "RSRP fuera de rango físico razonable"
assert 0 < REQ["rsrp_prob"] <= 1,          "rsrp_prob es fracción (0 a 1)"
assert 0 < REQ["sinr_prob"] <= 1,          "sinr_prob es fracción (0 a 1)"
assert REQ["thr_borde_ul_mbps"] < REQ["thr_borde_dl_mbps"], "¿UL mayor que DL?"

# términos de referencia (esto sí es el enunciado)
assert REQ["bw_hz"] <= 80e6,        "la licencia es de 80 MHz — no puedes usar más"
assert REQ["max_sitios"] <= 7,      "el municipio autoriza máximo 7 sitios"
assert REQ["p_tx_dbm_max"] <= 43.0, "la potencia máxima por sector es 43 dBm"

# coherencia de R5 con tu propio cálculo
assert None not in (kbps_por_abonado, demanda_mbps_km2), "completa el cálculo de R5"
assert REQ["capacidad_mbps_km2"] >= demanda_mbps_km2, \
    "R5 no cubre la demanda que tú mismo calculaste — el contrato nace roto"

print("REQ bien formado ✓ — la defensa de los valores es tuya (justificación)")


**JUSTIFICACIÓN — responde aquí mismo (edita esta celda):**

1. ¿De qué fila de la Tabla 0.1 sale tu umbral RSRP, y por qué esa fila
   traduce *"en todo Jesús María"*?

   _tu respuesta..._

2. La Tabla 0.3 dice que un stream HD pide 5–8 Mbps. ¿Por qué tu
   `thr_borde_dl_mbps` es distinto de ese número?

   _tu respuesta..._

3. ¿Qué margen dejaste entre la demanda calculada y tu R5? ¿Qué pasaría
   con tu diseño si la participación de mercado sube de 25% a 35%?

   _tu respuesta..._

4. ¿Por qué R6 (servicios y latencia) no aparece en el `REQ`?

   _tu respuesta..._


---

*(Fin de la Fase 0 — las Fases 1 a 6 se agregan tras validar esta dinámica.)*
